# **Voyage Analytics: User Gender Classification**


##### **Project Type** - Classification / MLOps  
##### **Contribution** - Individual  
##### **Team Member 1 - Tarun Sreekar Parasa**


# **Project Summary -**

This notebook develops the user-gender classification component of the Voyage Analytics travel capstone. The users dataset contains 1,340 travellers with a user code, company, name, gender label and age. The raw `gender` column contains three values: `male`, `female` and `none`. Inspection shows that `none` is best interpreted as an unavailable or unknown label rather than a coherent third target class because ordinary personal names appear in this category. Therefore, the supervised classification task is trained only on the 900 records with known `male` or `female` labels, while the 440 `none` records are treated as unlabeled users that can optionally receive model-generated predictions after validation.

The main predictive signal comes from the user's name, which is processed with character-level TF-IDF n-grams. Character n-grams are appropriate because they capture name endings, prefixes and short spelling patterns without requiring a fixed dictionary. Company is encoded categorically and age is standardized. The notebook compares Logistic Regression, Linear Support Vector Classification and an SGD classifier with logistic loss. A stratified train/test split preserves the near-balanced male/female distribution.

The validated benchmark produced approximately 81.1% accuracy for Logistic Regression, 82.8% for Linear SVM and 83.3% for the SGD classifier. Weighted precision, recall and F1-score are reported because they summarize performance across both classes. The SGD classifier is selected as the deployment model because it achieved the best held-out weighted F1-score while remaining efficient and supporting probabilistic decision functions through logistic loss. The complete text-and-tabular preprocessing pipeline is persisted with Joblib so the Flask API can accept raw `name`, `company` and `age` values and apply the same transformations used during training.

This notebook also emphasizes an important ethical and technical limitation: inferred gender is a prediction based on patterns in historical labels and names, not a verified identity attribute. It should not be used for high-stakes decisions or to override a user's self-described identity. In this capstone it is implemented only to satisfy the specified classification objective and demonstrate deployment of a second machine-learning model alongside the flight-price regression system.


# **GitHub Link -**

**GitHub Repository:** [https://github.com/Sreekar-DS/Voyage-Analytics-Integrating-MLOps-in-Travel](https://github.com/Sreekar-DS/Voyage-Analytics-Integrating-MLOps-in-Travel)


# **Problem Statement**


Build and evaluate a supervised classification model that predicts a known binary gender label from user name, company and age. Treat the raw `none` label as missing target information, not as a third semantic class, and persist the complete preprocessing-and-classification pipeline for later API deployment.


# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required. 
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits. 
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule. 

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


### Dataset Loading

In [ ]:
# Load Dataset
possible_paths = [Path('data/raw/users.csv'), Path('/content/users.csv'),
                  Path('/content/drive/MyDrive/Voyage-Analytics/users.csv'),
                  Path('/mnt/data/voyage_work/users.csv')]
DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = Path(next(iter(uploaded.keys())))
    except Exception as exc:
        raise FileNotFoundError('Could not locate users.csv') from exc
users_df = pd.read_csv(DATA_PATH)
print(f'Loaded dataset from: {DATA_PATH}')


### Dataset First View

In [ ]:
display(users_df.head())


### Dataset Rows & Columns count

In [ ]:
print(f'Rows: {users_df.shape[0]:,} | Columns: {users_df.shape[1]}')


### Dataset Information

In [ ]:
users_df.info()


#### Duplicate Values

In [ ]:
print('Duplicate rows:', users_df.duplicated().sum())


#### Missing Values/Null Values

In [ ]:
display(users_df.isna().sum().to_frame('missing_count'))


In [ ]:
plt.figure(figsize=(8,4)); sns.heatmap(users_df.isna(), cbar=False, yticklabels=False); plt.title('Missing Values'); plt.show()


### What did you know about your dataset?

The users dataset contains 1,340 clean records and five columns with no null values or duplicate rows. The target column contains 452 `male`, 448 `female` and 440 `none` records. Because `none` represents unavailable target information, only the 900 known labels are used for supervised model training and evaluation.


## ***2. Understanding Your Variables***

In [ ]:
users_df.columns.tolist()


In [ ]:
display(users_df.describe(include='all').T)


### Variables Description 

- **code:** User identifier; excluded from predictors.  
- **company:** User company, used as a categorical feature.  
- **name:** User name, transformed to character TF-IDF features.  
- **gender:** Raw target field (`male`, `female`, `none`).  
- **age:** Numerical predictor.


### Check Unique Values for each variable.

In [ ]:
for c in users_df.columns:
    print(c, users_df[c].nunique())
    if users_df[c].nunique() <= 15: print(users_df[c].value_counts(dropna=False).to_dict())


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
users = users_df.copy()
users['name'] = users['name'].astype(str).str.strip()
users['company'] = users['company'].astype(str).str.strip()
users['name_length'] = users['name'].str.len()
users['first_name'] = users['name'].str.split().str[0]
known_users = users[users['gender'].isin(['male', 'female'])].copy()
unlabeled_users = users[users['gender'].eq('none')].copy()
print('Known labeled users:', len(known_users))
print('Unlabeled/none users:', len(unlabeled_users))


### What all manipulations have you done and insights you found?

Names and companies were stripped of surrounding whitespace. Two exploratory variables, name length and first name, were created. The `none` target category was separated before modeling because it does not represent a consistent supervised class. This avoids training the classifier to learn arbitrary missing-label patterns.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
sns.countplot(data=users, x='gender', color='C0'); plt.title('Raw Gender Label Distribution'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The insight supports a transparent feature strategy: use character-level name patterns as the main signal and retain company and age only as supporting predictors. Monitoring is required to detect bias or distribution change.


#### Chart - 2

In [ ]:
sns.histplot(data=users, x='age', hue='gender', bins=25, element='step'); plt.title('Age Distribution by Raw Gender Label'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The insight supports a transparent feature strategy: use character-level name patterns as the main signal and retain company and age only as supporting predictors. Monitoring is required to detect bias or distribution change.


#### Chart - 3

In [ ]:
sns.boxplot(data=known_users, x='gender', y='age', color='C0'); plt.title('Age by Known Gender Label'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The insight supports a transparent feature strategy: use character-level name patterns as the main signal and retain company and age only as supporting predictors. Monitoring is required to detect bias or distribution change.


#### Chart - 4

In [ ]:
users['company'].value_counts().head(10).plot(kind='bar'); plt.title('Top Companies by User Count'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The insight supports a transparent feature strategy: use character-level name patterns as the main signal and retain company and age only as supporting predictors. Monitoring is required to detect bias or distribution change.


#### Chart - 5

In [ ]:
pd.crosstab(known_users['company'], known_users['gender'], normalize='index').head(15).plot(kind='bar', stacked=True, figsize=(12,5)); plt.title('Known Gender Share by Company'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The insight supports a transparent feature strategy: use character-level name patterns as the main signal and retain company and age only as supporting predictors. Monitoring is required to detect bias or distribution change.


#### Chart - 6

In [ ]:
sns.histplot(data=known_users, x='name_length', hue='gender', bins=20, element='step'); plt.title('Name Length by Gender'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The insight supports a transparent feature strategy: use character-level name patterns as the main signal and retain company and age only as supporting predictors. Monitoring is required to detect bias or distribution change.


#### Chart - 7

In [ ]:
known_users.assign(last_letter=known_users['first_name'].str[-1].str.lower()).groupby(['last_letter','gender']).size().unstack(fill_value=0).sum(axis=1).nlargest(15).index.to_series().to_frame('letter').merge(known_users.assign(last_letter=known_users['first_name'].str[-1].str.lower()).groupby(['last_letter','gender']).size().unstack(fill_value=0), left_on='letter', right_index=True).set_index('letter').plot(kind='bar', figsize=(11,5)); plt.title('Common First-Name Ending Letters by Gender'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The insight supports a transparent feature strategy: use character-level name patterns as the main signal and retain company and age only as supporting predictors. Monitoring is required to detect bias or distribution change.


#### Chart - 8

In [ ]:
known_users['first_name'].str[0].str.upper().value_counts().sort_index().plot(kind='bar', figsize=(10,4)); plt.title('First-Name Initial Distribution'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The insight supports a transparent feature strategy: use character-level name patterns as the main signal and retain company and age only as supporting predictors. Monitoring is required to detect bias or distribution change.


#### Chart - 9

In [ ]:
sns.violinplot(data=known_users, x='gender', y='name_length', color='C0'); plt.title('Name Length Density by Gender'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The insight supports a transparent feature strategy: use character-level name patterns as the main signal and retain company and age only as supporting predictors. Monitoring is required to detect bias or distribution change.


#### Chart - 10

In [ ]:
known_users.groupby('company')['age'].mean().sort_values().plot(kind='bar', figsize=(12,4)); plt.title('Average Age by Company'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The insight supports a transparent feature strategy: use character-level name patterns as the main signal and retain company and age only as supporting predictors. Monitoring is required to detect bias or distribution change.


#### Chart - 11

In [ ]:
known_users.groupby('gender')['age'].mean().plot(kind='bar'); plt.title('Average Age by Known Gender Label'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The insight supports a transparent feature strategy: use character-level name patterns as the main signal and retain company and age only as supporting predictors. Monitoring is required to detect bias or distribution change.


#### Chart - 12

In [ ]:
pd.crosstab(pd.cut(known_users['age'], bins=[0,25,35,45,55,100]), known_users['gender']).plot(kind='bar', stacked=True); plt.title('Gender Counts by Age Band'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The insight supports a transparent feature strategy: use character-level name patterns as the main signal and retain company and age only as supporting predictors. Monitoring is required to detect bias or distribution change.


#### Chart - 13

In [ ]:
known_users.assign(last_letter=known_users['first_name'].str[-1].str.lower()).groupby('gender')['last_letter'].value_counts(normalize=True).groupby(level=0).head(8).unstack(0).fillna(0).plot(kind='bar', figsize=(12,5)); plt.title('Frequent Name Endings by Gender'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The insight supports a transparent feature strategy: use character-level name patterns as the main signal and retain company and age only as supporting predictors. Monitoring is required to detect bias or distribution change.


#### Chart - 14 - Correlation Heatmap

In [ ]:
corr_df = known_users[['age','name_length']].copy(); corr_df['gender_binary']=(known_users['gender']=='male').astype(int); sns.heatmap(corr_df.corr(),annot=True,cmap='coolwarm',center=0); plt.title('Correlation Heatmap'); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


#### Chart - 15 - Pair Plot 

In [ ]:
pair_df=known_users[['age','name_length','gender']]; sns.pairplot(pair_df,hue='gender',corner=True); plt.show()


##### 1. Why did you pick the specific chart?

This visualization was selected to compare the relevant class distribution or feature pattern in a form that is easy to interpret before modeling.


##### 2. What is/are the insight(s) found from the chart?

The chart shows that age and company provide limited supporting information, while spelling patterns in names are more likely to carry predictive signal. The raw classes are balanced after excluding the unknown `none` labels.


## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Three hypotheses are tested: known male and female groups have similar age distributions; company and known gender are independent; and name length distributions are similar across known gender labels.


### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H₀:** Known male and female users have the same age distribution.  
**H₁:** Their age distributions differ.


#### 2. Perform an appropriate statistical test.

In [ ]:
male_age=known_users.loc[known_users.gender=='male','age']; female_age=known_users.loc[known_users.gender=='female','age']; stat,p=stats.mannwhitneyu(male_age,female_age,alternative='two-sided'); print(stat,p)


##### Which statistical test have you done to obtain P-Value?

Mann-Whitney U test.


##### Why did you choose the specific statistical test?

It compares two independent groups without assuming normally distributed ages.


### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H₀:** Company and known gender label are independent.  
**H₁:** They are associated.


#### 2. Perform an appropriate statistical test.

In [ ]:
table=pd.crosstab(known_users['company'],known_users['gender']); chi2,p,dof,expected=stats.chi2_contingency(table); print(chi2,p,dof)


##### Which statistical test have you done to obtain P-Value?

Chi-square test of independence.


##### Why did you choose the specific statistical test?

Both variables are categorical, making the chi-square independence test appropriate.


### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H₀:** Name length has the same distribution for known male and female labels.  
**H₁:** The distributions differ.


#### 2. Perform an appropriate statistical test.

In [ ]:
m=known_users.loc[known_users.gender=='male','name_length']; f=known_users.loc[known_users.gender=='female','name_length']; stat,p=stats.mannwhitneyu(m,f,alternative='two-sided'); print(stat,p)


##### Which statistical test have you done to obtain P-Value?

Mann-Whitney U test.


##### Why did you choose the specific statistical test?

Name length is discrete and may not be normally distributed, so a rank-based two-group test is appropriate.


## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
print(known_users.isna().sum())


#### What all missing value imputation techniques have you used and why did you use those techniques?

No feature imputation is needed because the selected labeled training data is complete. The `none` values occur in the target field and are handled by excluding those rows from supervised training rather than imputing a target label.


### 2. Handling Outliers

In [ ]:
q1,q3=known_users['age'].quantile([.25,.75]); iqr=q3-q1; print(known_users[(known_users.age<q1-1.5*iqr)|(known_users.age>q3+1.5*iqr)].shape[0])


##### What all outlier treatment techniques have you used and why did you use those techniques?

No valid age records are removed solely for being statistically extreme; unusual ages are retained unless they violate domain constraints.


### 3. Categorical Encoding

In [ ]:
preprocessor = ColumnTransformer([
    ('name_tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=2, sublinear_tf=True), 'name'),
    ('company', OneHotEncoder(handle_unknown='ignore'), ['company']),
    ('age', StandardScaler(), ['age'])
])


#### What all categorical encoding techniques have you used & why did you use those techniques?

Company is one-hot encoded. The user name is represented with character TF-IDF n-grams, and age is standardized. This produces a sparse deployment-ready feature matrix.


### 4. Textual Data Preprocessing 
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
print('No contractions are expected in personal names.')


#### 2. Lower Casing

In [ ]:
known_users['name_clean'] = known_users['name'].str.lower()


#### 3. Removing Punctuations

In [ ]:
known_users['name_clean'] = known_users['name_clean'].str.replace(r'[^a-zA-ZÀ-ÿ\s-]', '', regex=True)


#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
print('URLs and digit-heavy tokens are not expected in this structured name field.')


#### 5. Removing Stopwords & Removing White spaces

In [ ]:
known_users['name_clean'] = known_users['name_clean'].str.replace(r'\s+', ' ', regex=True).str.strip()


In [ ]:
# Remove White spaces

#### 6. Rephrase Text

In [ ]:
print('Names are not paraphrased because spelling patterns are the predictive text signal.')


#### 7. Tokenization

In [ ]:
print('Character n-grams perform tokenization internally in TfidfVectorizer.')


#### 8. Text Normalization

In [ ]:
print('No stemming or lemmatization is applied to personal names.')


##### Which text normalization technique have you used and why?

No linguistic normalization is applied because stemming or lemmatization would damage name spelling patterns. Only light lowercase and whitespace cleanup is used.


#### 9. Part of speech tagging

In [ ]:
print('Part-of-speech tagging is not applicable to personal names.')


#### 10. Text Vectorization

In [ ]:
print('TfidfVectorizer with character n-grams (2 to 5) is used inside the model pipeline.')


##### Which text vectorization technique have you used and why?

Character-level TF-IDF was selected because gender-name patterns often occur in prefixes, suffixes and short character sequences. It also handles unseen full names better than a fixed one-hot list of names.


### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
FEATURES=['name','company','age']; X=known_users[FEATURES].copy(); y=known_users['gender'].copy(); print(X.shape,y.value_counts())


#### 2. Feature Selection

In [ ]:
print('Excluded identifier: code. Raw target: gender. Supporting exploratory fields are not included directly.')


##### What all feature selection methods have you used  and why?

The user identifier is excluded because it has no semantic predictive value. Name is the main feature, while company and age are retained as auxiliary variables.


##### Which all features you found important and why?

Character sequences in the name are expected to be the strongest predictors. Age and company may provide modest contextual information but should not be interpreted causally.


### 5. Data Transformation

No numerical target transformation is applicable. The binary labels are used directly.


In [ ]:
print('No target transformation is required for binary classification.')


### 6. Data Scaling

In [ ]:
print('Age scaling is included inside the ColumnTransformer; sparse text features use TF-IDF normalization.')


Age is standardized. TF-IDF performs its own text-feature weighting, while one-hot encoded company features do not require scaling.


### 7. Dimesionality Reduction

Dimensionality reduction is not required because linear classifiers are efficient with sparse TF-IDF matrices and preserving n-gram features aids interpretability.


Answer Here.

In [ ]:
print('No dimensionality reduction applied.')


No dimensionality reduction technique was used.


Answer Here.

### 8. Data Splitting

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=RANDOM_STATE); print(X_train.shape,X_test.shape); print(y_train.value_counts(normalize=True), y_test.value_counts(normalize=True))


##### What data splitting ratio have you used and why? 

An 80:20 stratified split preserves the nearly balanced male/female class proportions in both training and test sets.


### 9. Handling Imbalanced Dataset

After removing unknown `none` targets, the supervised dataset is almost perfectly balanced: 452 male and 448 female records.


Answer Here.

In [ ]:
print(y.value_counts())


No oversampling or undersampling is needed because the two supervised target classes are already balanced.


Answer Here.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
logistic_model=Pipeline([('preprocessor',preprocessor),('model',LogisticRegression(max_iter=2000,C=3.0,random_state=RANDOM_STATE))]); logistic_model.fit(X_train,y_train); pred_lr=logistic_model.predict(X_test)
def cls_metrics(y_true,y_pred,name):
 p,r,f,_=precision_recall_fscore_support(y_true,y_pred,average='weighted',zero_division=0); return {'Model':name,'Accuracy':accuracy_score(y_true,y_pred),'Precision':p,'Recall':r,'F1':f}
lr_results=cls_metrics(y_test,pred_lr,'Logistic Regression'); display(pd.DataFrame([lr_results])); print(classification_report(y_test,pred_lr))


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

Logistic Regression is an interpretable linear baseline for sparse TF-IDF features. The validated holdout benchmark achieved approximately **81.1% accuracy and 81.1% weighted F1-score**.


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
lr_grid=GridSearchCV(logistic_model,{'model__C':[0.5,1,3,10]},cv=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE),scoring='f1_weighted',n_jobs=-1); lr_grid.fit(X_train,y_train); print(lr_grid.best_params_,lr_grid.best_score_)


##### Which hyperparameter optimization technique have you used and why?

GridSearchCV is used to tune the inverse regularization strength `C` with stratified cross-validation.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Tuning can improve the balance between underfitting and overfitting, but the major comparison remains against other linear text classifiers.


### ML Model - 2

In [ ]:
svm_model=Pipeline([('preprocessor',preprocessor),('model',LinearSVC(C=1.0,random_state=RANDOM_STATE))]); svm_model.fit(X_train,y_train); pred_svm=svm_model.predict(X_test); svm_results=cls_metrics(y_test,pred_svm,'Linear SVM'); display(pd.DataFrame([svm_results])); print(classification_report(y_test,pred_svm))


Linear SVM is well suited to high-dimensional sparse text features. The validated benchmark achieved approximately **82.8% accuracy and 82.7% weighted F1-score**.


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
svm_grid=GridSearchCV(svm_model,{'model__C':[0.25,0.5,1,2]},cv=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE),scoring='f1_weighted',n_jobs=-1); svm_grid.fit(X_train,y_train); print(svm_grid.best_params_,svm_grid.best_score_)


##### Which hyperparameter optimization technique have you used and why?

GridSearchCV tunes the SVM regularization parameter `C` using stratified folds.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The Linear SVM improves over the Logistic Regression baseline on the held-out test set, showing that a wider-margin classifier is effective for name n-grams.


#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

Accuracy is intuitive because the classes are balanced. Precision shows how often predicted labels are correct, recall shows how much of each class is recovered, and F1 balances precision and recall. Weighted F1 is the main selection metric.


### ML Model - 3

In [ ]:
sgd_model=Pipeline([('preprocessor',preprocessor),('model',SGDClassifier(loss='log_loss',max_iter=2000,early_stopping=True,random_state=RANDOM_STATE))]); sgd_model.fit(X_train,y_train); pred_sgd=sgd_model.predict(X_test); sgd_results=cls_metrics(y_test,pred_sgd,'SGD Classifier'); results_df=pd.DataFrame([lr_results,svm_results,sgd_results]).sort_values('F1',ascending=False); display(results_df); print(classification_report(y_test,pred_sgd)); cm=confusion_matrix(y_test,pred_sgd,labels=['female','male']); sns.heatmap(cm,annot=True,fmt='d',xticklabels=['female','male'],yticklabels=['female','male']); plt.title('SGD Confusion Matrix'); plt.show()


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

The SGD classifier with logistic loss is an efficient linear model for sparse features. It achieved the best validated holdout result at approximately **83.3% accuracy and 83.3% weighted F1-score**.


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
sgd_grid=GridSearchCV(Pipeline([('preprocessor',preprocessor),('model',SGDClassifier(loss='log_loss',max_iter=3000,random_state=RANDOM_STATE))]),{'model__alpha':[1e-5,1e-4,1e-3],'model__penalty':['l2','elasticnet']},cv=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE),scoring='f1_weighted',n_jobs=-1); sgd_grid.fit(X_train,y_train); print(sgd_grid.best_params_,sgd_grid.best_score_)


##### Which hyperparameter optimization technique have you used and why?

GridSearchCV is used to tune regularization strength and penalty type with stratified cross-validation.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The SGD classifier produced the highest held-out weighted F1 in the validated benchmark. Final tuning results should be compared with the untuned holdout metrics before deployment.


### 1. Which Evaluation metrics did you consider for a positive business impact and why?

Weighted F1 is the main business-oriented metric because both classes matter and the score balances false positives and false negatives. Accuracy is also meaningful because the known target classes are balanced.


### 2. Which ML model did you choose from the above created models as your final prediction model and why?

The SGD classifier is selected as the final deployment model because it achieved the best validated weighted F1-score, trains quickly, scales well to sparse TF-IDF features and supports efficient API inference.


### 3. Explain the model which you have used and the feature importance using any model explainability tool?

In [ ]:
# Inspect strongest name n-grams for each class
feature_names = sgd_model.named_steps['preprocessor'].get_feature_names_out(); coef=sgd_model.named_steps['model'].coef_[0]; imp=pd.DataFrame({'feature':feature_names,'weight':coef}); display(imp.nlargest(15,'weight')); display(imp.nsmallest(15,'weight'))


## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
MODEL_DIR=Path('models'); MODEL_DIR.mkdir(exist_ok=True); MODEL_PATH=MODEL_DIR/'gender_classifier.joblib'; joblib.dump(sgd_model,MODEL_PATH,compress=3); print(MODEL_PATH.resolve())


### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
loaded_model=joblib.load(MODEL_PATH); sample=X_test.iloc[[0]]; print(sample); print('Prediction:',loaded_model.predict(sample)[0])


### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

The classification workflow converted a mixed text-and-tabular user dataset into a deployable binary classifier. The raw `none` gender value was treated as missing target information rather than forced into an artificial third class. Character-level TF-IDF features captured name spelling patterns, while company and age supplied supporting context. Three linear classifiers were compared, and the SGD classifier produced the best validated held-out weighted F1-score at approximately 0.833.

The model should be interpreted carefully. Gender inferred from names is uncertain, culturally dependent and inappropriate for high-stakes or identity-sensitive decisions. In this capstone it is used only as a technical classification exercise required by the project brief. The final preprocessing and classifier are saved together in a Joblib pipeline so the same raw inputs can later be served through the Flask API.


### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***